# Transferring Topo stuff to python

In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
host=os.getenv('HOST')
host

## Set some paths and directories

In [ ]:
# ============================================================
# 6. Helper function for one panel
# ============================================================

def plot_ridgelet_panel(ax, fill_field, panel_title, ridgelet_length_fac=1.0, ridgelet_color_override=None, plot_ridgelets=True, alpha=0.5 ):
    """
    fill_field should be either raw or smoo
    """

    # Filled topography
    cf = ax.contourf(
        tlons[ip, yslice, xslice],
        tlats[ip, yslice, xslice],
        fill_field[ip, yslice, xslice],
        levels=topo_levels,
        cmap=cmap,
        norm=norm,
        alpha=alpha,
        transform=DataProj
    )

    plot_clngt = ridgelet_length_fac * clngt
    
    # Ridgelets colored by mxdis + local smoothed target_topo
    if plot_ridgelets == True:
        for i in iroo:
            for ir in range(maxridges):
    
                value = mxdis[i, ir] 
    
                if not np.all(np.isfinite([
                    latc[i, ir],
                    lonc[i, ir],
                    angll[i, ir],
                    clngt[i, ir],
                    value
                ])):
                    continue
                    
                
                lat0 = latc[i, ir]
                lon0 = lonc[i, ir]
    
                lat1, lon1, lat2, lon2 = Po.CAMridgelet(
                    lat=lat0,
                    lon=lon0,
                    angll=angll[i, ir],
                    clngt=plot_clngt[i, ir]
                )
    
                if ridgelet_color_override is None:
                    ridge_color = cmap(norm(value))
                else:
                    ridge_color = ridgelet_color_override
                    
                ax.plot(
                    [lon1, lon2],
                    [lat1, lat2],
                    color=ridge_color,
                    linewidth=ridge_linewidth,
                    transform=DataProj,
                    zorder=10
                )
    
    ax.set_extent(
        [lonrange[0], lonrange[1], latrange[0], latrange[1]],
        crs=DataProj
    )

    gl = ax.gridlines(
        draw_labels=True,
        linewidth=1,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )

    gl.top_labels   = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 11, 'color': 'gray'}
    gl.ylabel_style = {'size': 11, 'color': 'gray'}

    ax.set_title(panel_title, fontsize=13)

    return cf





In [ ]:
TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/
#Case = 'ne30pg3_Sco100_Sfi6_v11_test'
#Case = 'ne30pg3_Sco100_v8_test'
Case = 'ne30pg3_Sco100_Sfi6_v200_rch1p0_e02'
ogrid= 'ne30pg3'

# Do a file search in output directory
# Pick last (most recent) file
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_list*dat')
ListFile = foo[-1]


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/'+ogrid+'*.nc')
FileN = foo[-1]
print(FileN)
dCAM=xr.open_dataset( FileN )
print( list(dCAM.variables) )
print(np.shape(dCAM['lat']))
target_lon=dCAM['lon'].values
target_lat=dCAM['lat'].values
target_topo=dCAM['PHIS'].values / 9.8


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/topo*.nc')
FileN = foo[-1]
print(FileN)
topo=xr.open_dataset( FileN )
print(list(topo.variables))
print(topo.attrs)
raw=topo['terr_dev']+topo['terr_sm']
Loo = np.size(raw)
nc=int(np.sqrt( Loo/6 ))
print(int(nc))
raw=np.reshape( raw.values , (6,nc,nc) )
dev=np.reshape( topo['terr_dev'].values , (6,nc,nc) )
smoo=np.reshape( topo['terr_sm'].values , (6,nc,nc) )
tlats=np.reshape( topo['lat'].values , (6,nc,nc) )
tlons=np.reshape( topo['lon'].values , (6,nc,nc) )

# Define the range of indices
x_indices = np.arange(3000)
y_indices = np.arange(3000)

# Use meshgrid to create 2D arrays of x and y indices
xgrid, ygrid = np.meshgrid(x_indices, y_indices, indexing='xy')


foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/Ridge_tile_map.dat')
TileFile = foo[-1]
TileMap=Ti.rdtilemap( tile=TileFile, read_angll=True )

ntarget,maxridges = TileMap.MyObject.shape
print(ntarget,maxridges)
print( f"Tile shape {TileMap.lonc.shape}" )


latc=(180./np.pi)*TileMap.latc
lonc=(180./np.pi)*TileMap.lonc
angll=TileMap.angll
anglx=TileMap.anglx
aniso=TileMap.aniso
clngt=TileMap.LnObject # 
#clngt=TileMap.clngt
mxdis=TileMap.mxdis



In [ ]:
print(ListFile)

In [ ]:
aniso[aniso < 0.] = 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs


# ============================================================
# 1. Choose geographical region
# ============================================================

location = 'Colorado' # 'Mogollon'  

location_info = {

    'Colorado': {
        'panel': 4,
        'lonrange': np.asarray([-117., -100.]) + 360.,
        'latrange': np.asarray([30., 41.]),
        'projection': 'LambertConformal'
    },

    'Utah': {
        'panel': 4,
        'lonrange': np.asarray([-113., -108.]) + 360.,
        'latrange': np.asarray([36., 41.]),
        'projection': 'LambertConformal'
    },

    'Mogollon': {
        'panel': 4,
        'lonrange': np.asarray([-111., -110.]) + 360.,
        'latrange': np.asarray([34., 35.]),
        'projection': 'LambertConformal'
    },

    'Kaibab': {
        'panel': 4,
        'lonrange': np.asarray([-113., -111.]) + 360.,
        'latrange': np.asarray([36., 37.]),
        'projection': 'LambertConformal'
    },

    'SouthwestUS': {
        'panel': 4,
        'lonrange': np.asarray([-130., -104.]) + 360.,
        'latrange': np.asarray([29., 42.]),
        'projection': 'AlbersEqualArea'
    },

    'CONUS_South': {
        'panel': 4,
        'lonrange': np.asarray([-125., -70.]) + 360.,
        'latrange': np.asarray([24., 45.]),
        'projection': 'AlbersEqualArea'
    },

    'SouthAndesAntarcticPen': {
        'panel': 5,
        'lonrange': np.asarray([260., 300.]),
        'latrange': np.asarray([-75., -50.]),
        'projection': 'Orthographic'
    },

    'McMurdo': {
        'panel': 5,
        'lonrange': np.asarray([120., 230.]),
        'latrange': np.asarray([-87., -50.]),
        'projection': 'Orthographic'
    },

    'Antarctica': {
        'panel': 5,
        'lonrange': np.asarray([0., 360.]),
        'latrange': np.asarray([-90., -50.]),
        'projection': 'Orthographic'
    }
}


if location not in location_info:
    raise ValueError(f"Unknown location: {location}")

cfg = location_info[location]

panel    = cfg['panel']
ip       = panel - 1
lonrange = cfg['lonrange']
latrange = cfg['latrange']

central_longitude = np.average(lonrange)
central_latitude  = np.average(latrange)

In [ ]:
# ============================================================
# 2. Set map projection
# ============================================================

if cfg['projection'] == 'LambertConformal':

    MapProj = ccrs.LambertConformal(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'AlbersEqualArea':

    MapProj = ccrs.AlbersEqualArea(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

elif cfg['projection'] == 'Orthographic':

    if location == 'Antarctica':
        central_longitude = 180.
        central_latitude  = -90.

    MapProj = ccrs.Orthographic(
        central_longitude=central_longitude,
        central_latitude=central_latitude
    )

DataProj = ccrs.PlateCarree()


# ============================================================
# 3. Find target cells in region
# ============================================================

in_region = (
    (target_lon >  lonrange[0]) &
    (target_lon <= lonrange[1]) &
    (target_lat >  latrange[0]) &
    (target_lat <= latrange[1])
)

iroo = np.where(in_region)[0]

#iroo=iroo[0:1]
print(f"Number of target cells in region = {len(iroo)}")


# ============================================================
# 4. Find tile subsection needed for plot
# ============================================================

tile_mask = (
    (tlons[ip, :, :] >  lonrange[0]) &
    (tlons[ip, :, :] <= lonrange[1]) &
    (tlats[ip, :, :] >  latrange[0]) &
    (tlats[ip, :, :] <= latrange[1])
)

if not np.any(tile_mask):
    raise ValueError("Requested region does not overlap selected topo tile")

yy_idx, xx_idx = np.where(tile_mask)

xxrange = [xx_idx.min(), xx_idx.max()]
yyrange = [yy_idx.min(), yy_idx.max()]

print("x range =", xxrange)
print("y range =", yyrange)
print("central latitude =", central_latitude)

xslice = slice(xxrange[0], xxrange[1] + 1)
yslice = slice(yyrange[0], yyrange[1] + 1)


# ============================================================
# 5. Plot settings
# ============================================================

topo_levels = np.linspace(-1000., 1000., 21)
cmap = plt.get_cmap('terrain')

# Same mapping for shaded topo and ridgelet coloring
norm = mpl.colors.BoundaryNorm(
    topo_levels,
    ncolors=cmap.N,
    clip=True
)

ridge_linewidth = 1.8


# ============================================================
# 7. Create figure
# ============================================================

x_npanels = 3 
fig = plt.figure(figsize=( x_npanels*9.1 , 9))

#ax1 = fig.add_axes([0.05, 0.12, 0.40, 0.78], projection=MapProj)
#ax2 = fig.add_axes([0.52, 0.12, 0.40, 0.78], projection=MapProj)

x_LL, x_wd=0.03,0.28
ax1 = fig.add_axes([x_LL, 0.12, x_wd, 0.78], projection=MapProj)
x_LL=x_LL+0.33
ax2 = fig.add_axes([x_LL, 0.12, x_wd, 0.78], projection=MapProj)
x_LL=x_LL+0.33
ax3 = fig.add_axes([x_LL, 0.12, x_wd, 0.78], projection=MapProj)

cf1 = plot_ridgelet_panel(
    ax1,
    fill_field=dev,
    plot_ridgelets=False,
    panel_title="Dev topography shading "
)


cf2 = plot_ridgelet_panel(
    ax2,
    fill_field=0.5*dev,
    ridgelet_length_fac=1.,
    ridgelet_color_override=None,
    alpha=0.2,
    panel_title="Dev topography shading + \nRidgelet color = mxdis (drawn w/ clngt NOT LnObject)"
)

cf3 = plot_ridgelet_panel(
    ax3,
    fill_field=0.5*dev,
    ridgelet_length_fac=1. , #aniso,
    ridgelet_color_override='black',
    alpha=0.5,
    panel_title="Dev topography shading + \nRidgelets - black (drawn w/ aniso*clngt NOT LnObject)"
)


# ============================================================
# 8. Shared colorbar
# ============================================================


cax = fig.add_axes([0.99, 0.18, 0.02, 0.62])

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cax)
cbar.set_label('Elevation-like value (m)', fontsize=12)
# If you prefer to be more explicit:
# cbar.set_label('mxdis + target_topo (m)', fontsize=12)


# ============================================================
# 9. Suptitle
# ============================================================

fig.suptitle(
    "3000x3000 topography over-plotted with ridgelets\n"
    f"{TileFile}",
    fontsize=15,
    y=0.97
)

plt.show()

In [ ]:
# Do a file search in output directory
# Pick last (most recent) file
foo=glob.glob( TopoRoot +'/cases/'+ Case + '/output/remap*dat')
FileN = foo[-1]
print(FileN)
f=FortranFile( FileN, 'r')
idims=f.read_record( '<i4' )

nc=idims[0]

uniqi=f.read_record( '<f8'   ).reshape( 6,nc,nc)
aniso=f.read_record( '<f8'   ).reshape( 6,nc,nc)
anglx=f.read_record( '<f8'   ).reshape( 6,nc,nc)
mxdis=f.read_record( '<f8'   ).reshape( 6,nc,nc)
hwdth=f.read_record( '<f8'   ).reshape( 6,nc,nc)
clngt=f.read_record( '<f8'   ).reshape( 6,nc,nc)
block=f.read_record( '<f8'   ).reshape( 6,nc,nc)
profi=f.read_record( '<f8'   ).reshape( 6,nc,nc)
nodes=f.read_record( '<f8'   ).reshape( 6,nc,nc)
wedge=f.read_record( '<f8'   ).reshape( 6,nc,nc)
nodos=f.read_record( '<f8'   ).reshape( 6,nc,nc)
wedgo=f.read_record( '<f8'   ).reshape( 6,nc,nc)


In [ ]:
import rdremap as RM

Remp=RM.rdremap( FileN,  fields="all" )

In [ ]:
print( uniqi.max() )
print( Remp.uniqid.max() )


In [ ]:
plt.plot( TileMap.mxdis[iroo,:].flatten(),'.')
plt.ylim(0,2000)
plt.xlim(1300,1330)

print( TileMap.mxdis[iroo,:].flatten()[1320:1330] )
print( TileMap.MyObject[iroo,:].flatten()[1320:1330] )


In [ ]:
chid = 160310 # 160223
zoo = np.where( Remp.uniqid== chid )
poo = np.where( TileMap.MyObject == chid )

In [ ]:
zood=np.asarray(zoo)
print( zood.shape )
pood=np.asarray(poo)
print( pood[0] )
print( TileMap.MyObject [ pood[0], pood[1] ] )
print( TileMap.LnObject [ pood[0], pood[1] ] )
print( zood.shape[1] )
print( np.sum( TileMap.LnObject [ pood[0], pood[1] ] ))


In [ ]:
poop = TileMap.MyObject[iroo,:]
poop2=poop[ poop > 0] 
print( poop2.min(), poop2.max() )

test1 = np.zeros( poop2.max() - poop2.min() )
test2 = np.zeros( poop2.max() - poop2.min() )
print( test1.shape )
ii=0
#for chid in np.arange( poop2.min(), poop2.max() + 1 ):
for chid in np.arange( 160000,161000 + 1 ):
    zoo = np.where( Remp.uniqid== chid )
    boo = np.where( TileMap.MyObject == chid )
    zood=np.asarray(zoo)
    bood=np.asarray(boo)
    test1[ii]=zood.shape[1]
    test2[ii]= np.sum( TileMap.LnObject [ bood[0], bood[1] ] )
    ii=ii+1
    if ii%100==0:
        print(ii)


In [ ]:
npeaks = RdgList.npeaks

cube = RM.count_ids(Remp.uniqid, npeaks)          # cells per ID on the cube

v = TileMap.valid()                             # <-- essential
S = np.bincount(TileMap.MyObject[v].astype(int),
                weights=TileMap.LnObject[v],
                minlength=npeaks+1)[1:]

d = S - cube
print("ridges disagreeing :", int((d != 0).sum()), "of", npeaks)
print("totals             :", cube.sum(), "vs", S.sum())
print("max |difference|   :", np.abs(d).max())

# your original window, ID k -> index k-1
k = np.arange(160000, 161001)
print(np.column_stack([k, cube[k-1], S[k-1]])[:20])

In [ ]:
plt.plot( test1 - test2)
#plt.plot( test2 )


In [ ]:
npeaks = r.npeaks                       # from the ridge list

ids = uniqi.astype(np.int64).ravel()    # truncation, matching Fortran
ids = np.where(ids > 0, ids, 0)         # 0 is the ignore bin
cube_count = np.bincount(ids, minlength=npeaks+1)[1:]   # index -> ipk-1


In [ ]:
r2d=180./np.pi

print( iroo )
print( TileMap.MyObject[iroo,:] )
#print( TileMap.mxdis[iroo,:].astype(int) )
#print( TileMap.anglx[iroo,:].astype(int) )
#print( TileMap.angll[iroo,:].astype(int) )
print( TileMap.clngt[iroo,:].astype(int) )
#print( r2d*TileMap.latc[iroo,:] )
#print( r2d*TileMap.lonc[iroo,:] )
#print( TileMap.yloid[iroo,:].astype(int) )
#print( TileMap.xloid[iroo,:].astype(int) )

print( TileMap.LnObject[iroo,:]  ) #.astype(int) )



In [ ]:
from dataclasses import fields
[f.name for f in fields(TileMap)]

In [ ]:
plt.plot( TileMap.aniso[iroo,:].flatten(), '.' )

In [ ]:
print(TileFile)

TileFile0 = '/glade/work/juliob/Topo/NCARTopoJTB-fire//cases/ne30pg3_Sco100_Sfi6_v200_rch0p5/output/Ridge_tile_map.dat'

In [ ]:
r1 = Ti.rdtilemap(TileFile0, read_angll=True)
r2 = Ti.rdtilemap(TileFile,  read_angll=True)
v1, v2 = r1.valid(), r2.valid()
print(r1.LnObject[v1].mean(), r2.LnObject[v2].mean())   # did in-box crest grow?
print(r1.clngt[v1].mean(),    r2.clngt[v2].mean())      # should be identical
print((r1.clngt[v1] > 42).mean())                        # fraction ever truncated

In [ ]:
r2.clngt2.shape

In [ ]:
plt.plot(r2.clngt2,'.')

In [ ]:
import rdridgelist as RL

List=RL.rdridgelist( ListFile )

print( List.clngth.shape )

print( TileMap.LnObject.mean() )

In [ ]:
plt.scatter( RdgList.clngth, r2.clngt2, s=0.1  )
plt.xlabel( "Ridge List clngth" )
plt.ylabel( "Tile Map clngt2" )


In [ ]:
t=TileMap
r=RdgList
v   = t.valid()
ids = t.MyObject[v].astype(int)            # 1-based ridge IDs
S   = np.bincount(ids, weights=t.LnObject[v], minlength=r.npeaks+1)[1:]

def lis(a):                                 # length_in_square, vectorised
    ar = np.radians(np.asarray(a)); d = np.full(ar.shape, np.nan)
    m = (ar > -np.pi/4) & (ar <= np.pi/4);      d[m] = 1/np.cos(ar[m])
    m = (ar > np.pi/4) & (ar <= 3*np.pi/4);     d[m] = 1/np.cos(ar[m]-np.pi/2)
    m = (ar > 3*np.pi/4) & (ar <= 5*np.pi/4);   d[m] = 1/np.cos(ar[m]-np.pi)
    return d

nsw   = r.NSW
ncl   = np.minimum((r.clngth//2).astype(int), nsw)
drawn = 2*ncl + 1                            # cells in suba before rotation
pred  = drawn / lis(r.anglx)                 # cells after nearest-neighbour rotation

good = r.mxdis > 0
plt.scatter(pred[good], S[good], s=1, alpha=0.2)
plt.plot([0, pred[good].max()], [0, pred[good].max()], 'k-')
plt.xlabel('predicted painted crest cells'); plt.ylabel('sum(LnObject)')

In [ ]:
ratio = np.where(pred > 0, S/np.maximum(pred,1), np.nan)[good]
print("retained fraction: median %.2f  mean %.2f" % (np.nanmedian(ratio), np.nanmean(ratio)))
print("ridges retaining nothing: %.1f%%" % (100*np.nanmean(ratio < 0.05)))

# are the losers the unimportant ones?
plt.scatter(pred[good], S[good], s=2, c=np.log10(r.mxdis[good]), cmap='viridis')
plt.colorbar(label='log10 mxdis')

In [ ]:
m = good & (r.mxdis >= 1.0)          # paintridge2cube skips mxdis < 1
ratio = S[m] / np.maximum(pred[m], 1)
lm = np.log10(r.mxdis[m])

print("corr(log10 mxdis, retained) = %.3f" % np.corrcoef(lm, ratio)[0,1])
for lo, hi in [(1,10),(10,100),(100,1000),(1000,1e9)]:
    s = m & (r.mxdis>=lo) & (r.mxdis<hi)
    if s.sum():
        rr = S[s]/np.maximum(pred[s],1)
        print(f"  mxdis {lo:>6}-{hi:<6}: n={s.sum():6d}  median retained={np.median(rr):.2f}"
              f"  frac lost entirely={np.mean(rr<0.05):.2f}")